# 03. Preprocessing Pipeline & Principal Component Analysis (PCA)

This notebook benchmarks feature scaling strategies (StandardScaler vs RobustScaler) on log1p-transformed RFM features and evaluates dimensionality reduction via PCA.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from src.data.cleaner import clean_retail_data
from src.data.rfm_builder import build_rfm_features
from src.models.pipeline import build_rfm_preprocessor

## 1. Load Data & Fit Preprocessing Pipelines

In [ ]:
raw_df = pd.read_csv('data/online_retail_II.csv')
clean_df = clean_retail_data(raw_df)
rfm_df = build_rfm_features(clean_df)

pipe_std = build_rfm_preprocessor(scaler_type='standard')
X_std = pipe_std.fit_transform(rfm_df)

pipe_rob = build_rfm_preprocessor(scaler_type='robust')
X_rob = pipe_rob.fit_transform(rfm_df)

print('StandardScaler Mean:', np.round(X_std.mean(axis=0), 4))
print('StandardScaler Std:', np.round(X_std.std(axis=0), 4))
print('RobustScaler Median:', np.round(np.median(X_rob, axis=0), 4))

## 2. Principal Component Analysis (PCA)

In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_std)

pca_summary = pd.DataFrame({
    'Principal Component': ['PC1', 'PC2', 'PC3'],
    'Explained Variance Ratio': pca.explained_variance_ratio_,
    'Cumulative Variance Ratio': np.cumsum(pca.explained_variance_ratio_)
})
print(pca_summary)

components_df = pd.DataFrame(
    pca.components_,
    columns=['Recency_log', 'Frequency_log', 'Monetary_log'],
    index=['PC1', 'PC2', 'PC3']
)
print('\nPrincipal Component Loadings:')
print(components_df)